# Camada Silver — SISCAN / 10ª CRS

Padronização das nove tabelas Bronze para consumo pela Gold, no período de 2020 a 2025.

- Mantém os 11 municípios da regional e os metadados de origem.
- Converte matrizes de meses, anos e faixas etárias para formato longo.
- Preserva quantidades ausentes ou inválidas como nulas; não imputa zeros. Nas bases unidas por nome, descarta valores brutos nulos após o unpivot, conforme a regra de ingestão do TABNET.
- Valida a granularidade e os campos obrigatórios de todas as tabelas antes da gravação.

In [0]:
from collections.abc import Sequence, Mapping

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
ANO_INICIO, ANO_FIM = 2020, 2025
VALIDAR_REGRESSAO = True

CODIGOS_10CRS = (
    430040, 430187, 431060, 431171, 431175, 431530,
    431640, 431697, 431710, 431830, 432240,
)
MONTH_MAP = {
    "JANEIRO": 1, "FEVEREIRO": 2, "MARCO": 3, "MARO": 3,
    "ABRIL": 4, "MAIO": 5, "JUNHO": 6, "JULHO": 7,
    "AGOSTO": 8, "SETEMBRO": 9, "OUTUBRO": 10,
    "NOVEMBRO": 11, "DEZEMBRO": 12,
}
MES_MAP_ARQ = {"SETEMBR": 9, "OUTUBR": 10, "NOVEM": 11, "DEZEMB": 12}
REGEX_POP_ALVO = r"^(2[5-9]|3[0-9]|4[0-9]|5[0-9]|6[0-4])\s+a\s+\d+\s+anos"
REGEX_CATEGORIA = r"/(q5-.+?)_(?:anual|faixa_etaria)\.csv$"

COL_MUNICIPIO = "Munic.de residencia"
IDS_MUNICIPIO = ("codigo_municipio", "municipio")
COLS_ORIGEM = ("_file_source", "_ingestion_time")
COLS_QUANTIDADE = ("quantidade_raw", "quantidade")
COLS_RASTREIO = (*COLS_ORIGEM, "_bronze_source", "_silver_processed_time")
COLS_FAIXA = ("faixa_etaria_raw", "faixa_etaria", "fl_populacao_alvo")
COLS_TEMPO = ("ano", "mes", "dt_competencia")


def bronze_name(table_name: str) -> str:
    return f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}"


def silver_name(table_name: str) -> str:
    return f"{CATALOG}.{SCHEMA_SILVER}.{table_name}"


def quoted_col(name: str) -> Column:
    return F.col("`" + name.replace("`", "``") + "`")


def map_value(value: Column, mapping: Mapping[str, int]) -> Column:
    entries = [F.lit(item) for pair in mapping.items() for item in pair]
    return F.element_at(F.create_map(*entries), value)


def read_bronze(table_name: str) -> DataFrame:
    municipio = quoted_col(COL_MUNICIPIO)
    return (
        spark.table(bronze_name(table_name))
        .withColumn(
            "codigo_municipio",
            F.expr("try_cast(substring(trim(`Munic.de residencia`), 1, 6) as int)"),
        )
        .withColumn(
            "municipio",
            F.initcap(F.regexp_replace(F.trim(F.substring(municipio, 7, 1000)), r"\\", "")),
        )
        .filter(F.col("codigo_municipio").isin(*CODIGOS_10CRS))
    )


def file_year(df: DataFrame, pattern: str = r"([0-9]{4})\.csv$") -> DataFrame:
    year = F.regexp_extract(F.col("_file_source"), pattern, 1)
    return df.withColumn("ano", F.when(year != "", year.cast("int")))


def month_columns(df: DataFrame) -> list[str]:
    return [name for name in df.columns if "/" in name and name != COL_MUNICIPIO]


def age_columns(df: DataFrame, extra_ids: Sequence[str] = ()) -> list[str]:
    excluded = {COL_MUNICIPIO, *IDS_MUNICIPIO, *COLS_ORIGEM, *extra_ids}
    return [
        name for name in df.columns
        if name not in excluded and name.strip().casefold() != "total"
    ]


def unpivot_counts(
    df: DataFrame,
    values: Sequence[str],
    variable: str,
    extra_ids: Sequence[str] = (),
    *,
    drop_missing: bool = True,
    force_string: bool = False,
) -> DataFrame:
    if not values:
        raise ValueError(f"Nenhuma coluna de valores encontrada para {variable}.")

    if force_string:
        df = df.withColumns({name: quoted_col(name).cast("string") for name in values})

    result = df.unpivot(
        ids=[*IDS_MUNICIPIO, *extra_ids, *COLS_ORIGEM],
        values=[quoted_col(name) for name in values],
        variableColumnName=variable,
        valueColumnName="quantidade_raw",
    )
    # A união por nome da Bronze introduz nulos para colunas omitidas pelo TABNET.
    return result.filter(F.col("quantidade_raw").isNotNull()) if drop_missing else result


def with_month_period(df: DataFrame) -> DataFrame:
    month = F.upper(F.regexp_replace(F.split("mes_ano_col", "/").getItem(0), "[^A-Za-z]", ""))
    return (
        df.withColumn("ano", F.expr("try_cast(regexp_extract(mes_ano_col, '/([0-9]{4})$', 1) as int)"))
        .withColumn("mes", map_value(month, MONTH_MAP))
    )


def with_competence(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "dt_competencia",
        F.expr("try_to_date(concat_ws('-', ano, lpad(mes, 2, '0'), '01'), 'yyyy-MM-dd')"),
    )


def with_age_group(df: DataFrame) -> DataFrame:
    return (
        df.withColumn("faixa_etaria", F.regexp_replace(F.trim("faixa_etaria_raw"), r"^Entre\s+", ""))
        .withColumn("fl_populacao_alvo", F.col("faixa_etaria").rlike(REGEX_POP_ALVO))
    )


def finish_table(df: DataFrame, bronze_table: str, dimensions: Sequence[str]) -> DataFrame:
    return (
        df.withColumn("quantidade", F.expr("try_cast(quantidade_raw as int)"))
        .withColumn("_bronze_source", F.lit(bronze_name(bronze_table)))
        .withColumn("_silver_processed_time", F.current_timestamp())
        .select(*IDS_MUNICIPIO, *dimensions, *COLS_QUANTIDADE, *COLS_RASTREIO)
    )


def monthly_table(
    source: str,
    df: DataFrame,
    extra_ids: Sequence[str] = (),
    *,
    drop_missing: bool = True,
    force_string: bool = False,
) -> DataFrame:
    long = unpivot_counts(
        df, month_columns(df), "mes_ano_col", extra_ids,
        drop_missing=drop_missing, force_string=force_string,
    )
    return finish_table(
        with_competence(with_month_period(long)), source, (*extra_ids, *COLS_TEMPO)
    )


def age_table(
    source: str,
    df: DataFrame,
    extra_ids: Sequence[str] = (),
    *,
    monthly: bool = False,
    excluded: Sequence[str] = (),
) -> DataFrame:
    long = unpivot_counts(
        df, age_columns(df, (*extra_ids, *excluded)), "faixa_etaria_raw", extra_ids
    )
    result = with_age_group(long)
    dimensions = (*extra_ids, *COLS_FAIXA)
    if monthly:
        result = with_competence(result)
        dimensions = (*extra_ids, "dt_competencia", *COLS_FAIXA)
    return finish_table(result, source, dimensions)


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")

## 1. Exames gerais

`bronze.exames_geral_raw` → `silver.exames_geral`. Granularidade: município × ano × mês. Mantém as competências disponíveis na fonte, inclusive valores brutos nulos.

In [0]:
source = "exames_geral_raw"
df_exames_geral = monthly_table(source, read_bronze(source), drop_missing=False)

## 2. Faixa etária

`bronze.faixa_etaria_raw` → `silver.faixa_etaria`. Granularidade: município × ano × faixa etária.

In [0]:
source = "faixa_etaria_raw"
df_faixa_etaria = age_table(source, file_year(read_bronze(source)), ("ano",))

## 3. Adequabilidade

`bronze.adequabilidade_raw` → `silver.adequabilidade`. Granularidade: município × categoria de adequabilidade × ano × mês.

In [0]:
source = "adequabilidade_raw"
file_name = F.lower(F.col("_file_source"))
df_adequabilidade_base = read_bronze(source).withColumn(
    "categoria_adequabilidade",
    F.when(file_name.contains("insatisfat"), F.lit("Insatisfatoria"))
    .when(file_name.contains("satisfat"), F.lit("Satisfatoria"))
    .when(file_name.contains("rejeitada"), F.lit("Rejeitada"))
    .otherwise(F.lit("Desconhecida")),
)
df_adequabilidade = monthly_table(
    source, df_adequabilidade_base, ("categoria_adequabilidade",), force_string=True
)

## 4. Resultados citopatológicos

`bronze.resultados_raw` → `silver.resultados_citopatologicos`. Granularidade: município × ano × grupo de resultado × faixa etária.

In [0]:
source = "resultados_raw"
df_resultados_base = file_year(read_bronze(source)).withColumn(
    "grupo_resultado",
    F.regexp_extract(F.col("_file_source"), r"([^/]+?)_[0-9]{4}\.csv", 1),
)
df_resultados_citopatologicos = age_table(
    source, df_resultados_base, ("ano", "grupo_resultado")
)

## 4.1. Categorias citopatológicas

As extrações adicionais são mantidas em quatro tabelas, sem misturar granularidades:

- `resultados_categoria_mensal`: município × categoria × ano × mês.
- `resultados_categoria_anual`: município × categoria × ano.
- `resultados_categoria_faixa_etaria`: município × categoria × faixa etária.
- `resultados_alterados_consolidado`: município × faixa etária.

As duas últimas representam o período consolidado de 2020 a 2025, sem dimensão anual.

In [0]:
def category_base(source: str) -> DataFrame:
    category = F.regexp_extract(F.col("_file_source"), REGEX_CATEGORIA, 1)
    return read_bronze(source).withColumn(
        "categoria",
        F.regexp_replace(F.regexp_replace(category, "%20", " "), r"^q5-", ""),
    )


source = "resultados_categoria_mensal_raw"
df_resultados_categoria_mensal = monthly_table(source, category_base(source), ("categoria",))

source = "resultados_categoria_anual_raw"
df_category_annual = category_base(source)
year_cols = [name for name in df_category_annual.columns if name.isdigit()]
df_annual_long = unpivot_counts(df_category_annual, year_cols, "ano_raw", ("categoria",))
df_resultados_categoria_anual = finish_table(
    df_annual_long.withColumn("ano", F.expr("try_cast(ano_raw as int)")),
    source, ("categoria", "ano"),
)

source = "resultados_categoria_faixa_etaria_raw"
df_resultados_categoria_faixa_etaria = age_table(source, category_base(source), ("categoria",))

source = "resultados_alterados_consolidado_raw"
df_resultados_alterados_consolidado = age_table(source, read_bronze(source))

## 4.2. Outubro Rosa por faixa etária

`bronze.outubro_rosa_faixa_etaria_raw` → `silver.outubro_rosa_faixa_etaria`. Granularidade: município × ano × mês × faixa etária, de setembro a dezembro.

Os nomes dos arquivos usam abreviações de meses. A cobertura e a quantidade de registros dependem dos cabeçalhos presentes nas extrações. A auditoria de 17/11/2026 registrou 3.960 linhas e 67.925 exames na tabela publicada; a validação de regressão verifica se a nova transformação reproduz os dados existentes.

In [0]:
source = "outubro_rosa_faixa_etaria_raw"
month_name = F.upper(F.regexp_extract(F.col("_file_source"), r"/([A-Za-z]+)_[0-9]{4}", 1))

df_outubro_rosa_base = (
    file_year(read_bronze(source), r"_([0-9]{4})")
    .withColumn("mes", map_value(month_name, MES_MAP_ARQ))
)
df_outubro_rosa_faixa = age_table(
    source, df_outubro_rosa_base, ("ano", "mes"), monthly=True
)

## 5. Validação e persistência

Todas as tabelas devem possuir registros, campos obrigatórios preenchidos, chaves únicas e dimensões temporais válidas antes de qualquer gravação. Quantidades nulas são contabilizadas, não imputadas. O overwrite Delta substitui os dados sem excluir previamente a tabela.

A validação de regressão compara os dados analíticos gerados com as tabelas existentes, excluindo os metadados técnicos. Desative `VALIDAR_REGRESSAO` apenas quando houver uma atualização intencional da fonte ou da regra de limpeza.

In [0]:
from dataclasses import dataclass


@dataclass(frozen=True)
class SilverTable:
    name: str
    dataframe: DataFrame
    key: tuple[str, ...]
    required: tuple[str, ...]


def table_spec(
    name: str, df: DataFrame, key: Sequence[str], additional: Sequence[str] = ()
) -> SilverTable:
    return SilverTable(name, df, tuple(key), ("municipio", *key, *additional))


tables_to_validate = [
    table_spec("exames_geral", df_exames_geral, ("codigo_municipio", "ano", "mes"), ("dt_competencia",)),
    table_spec("faixa_etaria", df_faixa_etaria, ("codigo_municipio", "ano", "faixa_etaria")),
    table_spec(
        "adequabilidade", df_adequabilidade,
        ("codigo_municipio", "categoria_adequabilidade", "ano", "mes"), ("dt_competencia",),
    ),
    table_spec(
        "resultados_citopatologicos", df_resultados_citopatologicos,
        ("codigo_municipio", "ano", "faixa_etaria", "grupo_resultado"),
    ),
    table_spec(
        "resultados_categoria_mensal", df_resultados_categoria_mensal,
        ("codigo_municipio", "categoria", "ano", "mes"), ("dt_competencia",),
    ),
    table_spec(
        "resultados_categoria_anual", df_resultados_categoria_anual,
        ("codigo_municipio", "categoria", "ano"),
    ),
    table_spec(
        "resultados_categoria_faixa_etaria", df_resultados_categoria_faixa_etaria,
        ("codigo_municipio", "categoria", "faixa_etaria"),
    ),
    table_spec(
        "resultados_alterados_consolidado", df_resultados_alterados_consolidado,
        ("codigo_municipio", "faixa_etaria"),
    ),
    table_spec(
        "outubro_rosa_faixa_etaria", df_outubro_rosa_faixa,
        ("codigo_municipio", "ano", "mes", "faixa_etaria"), ("dt_competencia",),
    ),
]


def count_when(condition: Column) -> Column:
    return F.count(F.when(condition, 1))


def validate_table(spec: SilverTable) -> tuple[dict, list[str]]:
    df = spec.dataframe
    metrics = [
        F.count("*").alias("linhas"),
        F.sum("quantidade").alias("total_exames"),
        count_when(F.col("quantidade").isNull()).alias("quantidades_nulas"),
        count_when(F.col("quantidade") < 0).alias("quantidades_negativas"),
    ]
    for name in spec.required:
        col = F.col(name)
        metrics.append(
            count_when(col.isNull() | (F.trim(col.cast("string")) == "")).alias(f"nulos_{name}")
        )

    if "ano" in df.columns:
        metrics.append(count_when(~F.col("ano").between(ANO_INICIO, ANO_FIM)).alias("anos_invalidos"))
    if "mes" in df.columns:
        valid_months = range(9, 13) if spec.name == "outubro_rosa_faixa_etaria" else range(1, 13)
        metrics.append(count_when(~F.col("mes").isin(*valid_months)).alias("meses_invalidos"))
    if "categoria_adequabilidade" in df.columns:
        metrics.append(count_when(F.col("categoria_adequabilidade") == "Desconhecida").alias("categorias_invalidas"))

    summary = df.agg(*metrics).first().asDict()
    summary["chaves_duplicadas"] = df.groupBy(*spec.key).count().filter(F.col("count") > 1).count()

    diagnostic_only = {"linhas", "total_exames", "quantidades_nulas"}
    errors = [f"{name}={value}" for name, value in summary.items() if name not in diagnostic_only and value]
    if summary["linhas"] == 0:
        errors.append("tabela vazia")
    return summary, errors


def validate_regression(spec: SilverTable) -> list[str]:
    target = silver_name(spec.name)
    if not spark.catalog.tableExists(target):
        return []

    columns = [
        name for name in spec.dataframe.columns
        if name not in (*COLS_RASTREIO, "quantidade_raw")
    ]
    current = spark.table(target).select(*columns)
    proposed = spec.dataframe.select(*columns)
    removed = current.exceptAll(proposed).count()
    added = proposed.exceptAll(current).count()
    if removed or added:
        return [f"regressão: {removed} registros removidos/alterados, {added} novos/alterados"]
    return []


validation_errors = []
row_counts = {}
for spec in tables_to_validate:
    summary, errors = validate_table(spec)
    if VALIDAR_REGRESSAO and not errors:
        errors.extend(validate_regression(spec))
    row_counts[spec.name] = summary["linhas"]
    print(
        f"{spec.name}: {summary['linhas']} linhas | "
        f"total={summary['total_exames']} | nulos={summary['quantidades_nulas']} | "
        f"{'INVÁLIDA' if errors else 'OK'}"
    )
    if errors:
        validation_errors.append(f"{spec.name}: {', '.join(errors)}")

if validation_errors:
    raise ValueError("Falha na validação Silver:\n" + "\n".join(validation_errors))

for spec in tables_to_validate:
    (
        spec.dataframe.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_name(spec.name))
    )
    print(f"{spec.name}: {row_counts[spec.name]} linhas gravadas.")

display(
    spark.table(silver_name("exames_geral"))
    .select(*IDS_MUNICIPIO)
    .distinct()
    .orderBy("codigo_municipio")
)